In [7]:
import pandas as pd
import re

# ========= 1. Read raw data =========
df = pd.read_csv("metadataset.csv")

print("Number of original data records:", len(df))
print("Column names:", df.columns.tolist())

# ========= 2. Basic cleaning =========
# Only the Text type + English is kept
base = df[
    (df["Type"].fillna("").str.lower() == "text") &
    (df["Language"].fillna("").str.contains(r"\ben\b", case=False, na=False))
].copy()

print("The number of bars after basic screening:", len(base))

# ========= 3. Define Yes keyword =========
yes_keywords = [
    "children",
    "children's stories",
    "juvenile",
    "fairy tales",
    "fiction",
    "literature",
    "stories",
    "readers",
    "education",
    "school",
    "short stories",
    "novel",
    "poetry",
    "drama",
    "essays",
    "literary collections"
]

yes_pattern = "|".join(re.escape(k) for k in yes_keywords)

# ========= 4. Define Excluded Keywords =========
exclude_keywords = [
    "bibliography",
    "catalog",
    "catalogue",
    "dictionary",
    "index",
    "gazetteer",
    "periodical",
    "yearbook",
    "almanac",
    "encyclopedia",
    "encyclopaedia",
    "manual",
    "handbook",
    "law",
    "legislation",
    "statutes",
    "court",
    "regulations",
    "directory"
]

exclude_pattern = "|".join(re.escape(k) for k in exclude_keywords)

# ========= 5. Filter Yes across multiple fields =========
# Check Subjects / Bookshelves / Title first
yes_candidates = base[
    base["Subjects"].fillna("").str.contains(yes_pattern, case=False, na=False) |
    base["Bookshelves"].fillna("").str.contains(yes_pattern, case=False, na=False) |
    base["Title"].fillna("").str.contains(yes_pattern, case=False, na=False)
].copy()

print("Yes Number of candidates (not excluded):", len(yes_candidates))

# ========= 6. Exclude obviously unsuitable content =========
yes_final = yes_candidates[
    ~yes_candidates["Subjects"].fillna("").str.contains(exclude_pattern, case=False, na=False) &
    ~yes_candidates["Bookshelves"].fillna("").str.contains(exclude_pattern, case=False, na=False) &
    ~yes_candidates["Title"].fillna("").str.contains(exclude_pattern, case=False, na=False)
].copy()

print("Final number of Yes entries:", len(yes_final))

# ========= 7. Deduplication =========
yes_final = yes_final.drop_duplicates(subset=["Text#"])

print("The number of Yes bars after deduplication:", len(yes_final))

# ========= 8. Export Results =========
yes_final.to_csv("gutenberg_yes_candidates_full.csv", index=False)

yes_final[["Text#", "Title", "Authors", "Language", "Subjects", "Bookshelves"]].to_csv(
    "gutenberg_yes_candidates_small.csv", index=False
)

print("Export:")
print("- gutenberg_yes_candidates_full.csv")
print("- gutenberg_yes_candidates_small.csv")

# ========= 9. Check 20 results =========
print("\nThe first 20 results:")
print(
    yes_final[["Text#", "Title", "Authors", "Subjects", "Bookshelves"]]
    .head(20)
    .to_string(index=False)
)

原始数据条数: 78243
列名: ['Text#', 'Type', 'Issued', 'Title', 'Language', 'Authors', 'Subjects', 'LoCC', 'Bookshelves']
基础筛选后条数: 61015
Yes 候选条数（未排除）: 40223
最终 Yes 条数: 36242
去重后 Yes 条数: 36242
已导出:
- gutenberg_yes_candidates_full.csv
- gutenberg_yes_candidates_small.csv

前 20 条结果:
 Text#                                                                                                         Title                                                                  Authors                                                                                                                                                                                                                                                               Subjects                                                                                                                                                                         Bookshelves
     3                                                                           John F. Kenne

In [14]:
import os
import re
import time
import pandas as pd
import requests
from bs4 import BeautifulSoup

# ========= 1. Configuration =========
CSV_PATH = "gutenberg_yes_candidates_small.csv"
OUTPUT_DIR = "gutenberg_yes_texts"
MAX_DOWNLOADS = 300        
SLEEP_SECONDS = 1.5       

os.makedirs(OUTPUT_DIR, exist_ok=True)

headers = {
    "User-Agent": "Mozilla/5.0 (compatible; FIT5120-project/1.0)"
}

# ========= 2. Read candidate book list =========
df = pd.read_csv(CSV_PATH)

# Ensure there is a Text# column.
df = df[df["Text#"].notna()].copy()
df["Text#"] = df["Text#"].astype(int)

# Deduplication
df = df.drop_duplicates(subset=["Text#"])

# First, only take the first MAX_DOWNLOADS.
df = df.head(MAX_DOWNLOADS)

print(f"Ready to downloads {len(df)} books")

# ========= 3. Filename cleaning function =========
def safe_filename(text):
    text = str(text)
    text = re.sub(r'[\\/*?:"<>|]', "_", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text[:120]

# ========= 4. Find the txt download link =========
def find_txt_link(book_id):
    url = f"https://www.gutenberg.org/ebooks/{book_id}"
    r = requests.get(url, headers=headers, timeout=30)
    r.raise_for_status()

    soup = BeautifulSoup(r.text, "html.parser")

    # Prioritize Plain Text UTF-8
    for a in soup.find_all("a", href=True):
        text = a.get_text(" ", strip=True).lower()
        href = a["href"]

        if "plain text utf-8" in text:
            if href.startswith("/"):
                return "https://www.gutenberg.org" + href
            elif href.startswith("http"):
                return href

    # Next, find all the TXT links.
    for a in soup.find_all("a", href=True):
        href = a["href"]
        if ".txt" in href:
            if href.startswith("/"):
                return "https://www.gutenberg.org" + href
            elif href.startswith("http"):
                return href

    return None

# ========= 5. Download txt =========
def download_txt(url, save_path):
    r = requests.get(url, headers=headers, timeout=60)
    r.raise_for_status()

    if r.encoding is None:
        r.encoding = "utf-8"

    with open(save_path, "w", encoding="utf-8", errors="ignore") as f:
        f.write(r.text)

# ========= 6. Batch download =========
success = []
failed = []

for _, row in df.iterrows():
    book_id = int(row["Text#"])
    title = safe_filename(row.get("Title", f"book_{book_id}"))
    save_path = os.path.join(OUTPUT_DIR, f"{book_id}_{title}.txt")

    if os.path.exists(save_path):
        print(f"[Skip] Already exists: {save_path}")
        success.append(book_id)
        continue

    try:
        print(f"[Processing] {book_id} - {title}")

        txt_link = find_txt_link(book_id)

        if not txt_link:
            print("  -> No TXT download link found")
            failed.append((book_id, "No txt link found"))
            continue

        print(f"  -> Download link: {txt_link}")
        download_txt(txt_link, save_path)

        print("  -> Download Success")
        success.append(book_id)

        time.sleep(SLEEP_SECONDS)

    except Exception as e:
        print(f"  -> Download failed: {e}")
        failed.append((book_id, str(e)))

# ========= 7. Export Dairy =========
pd.DataFrame({"success_book_id": success}).to_csv("download_success.csv", index=False)
pd.DataFrame(failed, columns=["book_id", "error"]).to_csv("download_failed.csv", index=False)

print("\nDownload Complete")
print(f"Success: {len(success)}")
print(f"Fail: {len(failed)}")
print(f"Download Folder: {OUTPUT_DIR}")

准备下载 300 本书
[跳过] 已存在: gutenberg_yes_texts/3_John F. Kennedy's Inaugural Address.txt
[跳过] 已存在: gutenberg_yes_texts/4_Lincoln's Gettysburg Address Given November 19, 1863 on the battlefield near Gettysburg, Pennsylvania, USA.txt
[跳过] 已存在: gutenberg_yes_texts/6_Give Me Liberty or Give Me Death.txt
[跳过] 已存在: gutenberg_yes_texts/8_Abraham Lincoln's Second Inaugural Address.txt
[跳过] 已存在: gutenberg_yes_texts/9_Abraham Lincoln's First Inaugural Address.txt
[跳过] 已存在: gutenberg_yes_texts/10_The King James Version of the Bible.txt
[跳过] 已存在: gutenberg_yes_texts/11_Alice's Adventures in Wonderland.txt
[跳过] 已存在: gutenberg_yes_texts/12_Through the Looking-Glass.txt
[跳过] 已存在: gutenberg_yes_texts/13_The Hunting of the Snark_ An Agony in Eight Fits.txt
[跳过] 已存在: gutenberg_yes_texts/15_Moby-Dick; or, The Whale.txt
[跳过] 已存在: gutenberg_yes_texts/16_Peter Pan.txt
[跳过] 已存在: gutenberg_yes_texts/19_The Song of Hiawatha.txt
[跳过] 已存在: gutenberg_yes_texts/20_Paradise Lost.txt
[跳过] 已存在: gutenberg_yes_texts/21_Thre

In [16]:
import shutil

# Compress the entire folder into a zip file.
shutil.make_archive("gutenberg_yes_texts_bundle", "zip", "gutenberg_yes_texts")

print("Compress Done：gutenberg_yes_texts_bundle.zip")

压缩完成：gutenberg_yes_texts_bundle.zip
